In [ ]:
import boto3
import json

# Task 4: Creating an AWS Lambda function

Bedrock and Lambda are two tools that you can pair together that compliment each other well. While LLMs are very powerful, they are unable to do everything you need them to do. Bedrock is able to reach out to Lambda to call functions on your behalf. This allows you to "build" in new features to Bedrock.

In this task, you'll be creating a Lambda function that performs mathematical operations. In a future task, this will be utilized by Bedrock.

Open the AWS console and sign in using your exerciseuser credentials. 

**Note:** These are the same credentials that you created in the first exercise of this series.

At the top of the AWS Management Console, in the search bar, search for and choose Lambda.

Choose Create function.

Select Author from scratch.

For Function name, enter math-function.

Under Runtime, select Python 3.

Choose Create function.

Under Code source, select the existing code and delete it.

Paste the following code into the function:

In [ ]:
# Lambda function code
lambda_code = """import json

def add(num1, num2):
    return num1 + num2

def subtract(num1, num2):
    return num1 - num2

def multiply(num1, num2):
    return num1 * num2

def divide(num1, num2):
    if num2 == 0:
        raise ValueError("division by zero")
    return num1 / num2

def power(num1, num2):
    return num1 ** num2

def lambda_handler(event, context):
    operation = event.get('operation')
    try:
        num1 = float(event['num1'])
        num2 = float(event['num2'])
    except (KeyError, ValueError):
        return {
            'statusCode': 400,
            'body': json.dumps({'error': 'Invalid input, num1 and num2 must be numeric'})
        }

    # Map operations to functions
    operations = {
        'add': add,
        'subtract': subtract,
        'multiply': multiply,
        'divide': divide,
        'power': power
    }

    # Find the operation function and execute
    operation_func = operations.get(operation)
    
    if operation_func:
        try:
            result = operation_func(num1, num2)
            return {
                'statusCode': 200,
                'body': json.dumps({'result': result})
            }
        except ValueError as e:
            return {
                'statusCode': 400,
                'body': json.dumps({'error': str(e)})
            }
    else:
        return {
            'statusCode': 400,
            'body': json.dumps({'error': 'invalid operation'})
        }
"""

# Create deployment package
import zipfile
import io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zip_file:
    zip_file.writestr('lambda_function.py', lambda_code)
zip_buffer.seek(0)

# Get account ID
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']

# Create IAM client
iam_client = boto3.client('iam')

# Create IAM role for Lambda execution
role_name = 'lambda-execution-role'
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "lambda.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

try:
    # Try to create the role
    response = iam_client.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Execution role for Lambda functions'
    )
    print(f"✅ IAM role '{role_name}' created successfully!")
    
    # Attach basic execution policy
    iam_client.attach_role_policy(
        RoleName=role_name,
        PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
    )
    print(f"✅ Attached AWSLambdaBasicExecutionRole policy to '{role_name}'")
    
    # Wait a moment for the role to propagate
    import time
    time.sleep(2)
    
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"ℹ️  IAM role '{role_name}' already exists, using existing role.")
except Exception as e:
    print(f"⚠️  Error creating IAM role: {e}")
    print("Continuing with deployment attempt...")

# Get the role ARN
try:
    role_response = iam_client.get_role(RoleName=role_name)
    role_arn = role_response['Role']['Arn']
    print(f"✅ Using role ARN: {role_arn}")
except Exception as e:
    print(f"❌ Error getting role ARN: {e}")
    role_arn = f'arn:aws:iam::{account_id}:role/{role_name}'

# Create Lambda client
lambda_client = boto3.client('lambda')

# Deploy the Lambda function
function_name = 'math-function'

try:
    # Try to create the function
    zip_buffer.seek(0)
    response = lambda_client.create_function(
        FunctionName=function_name,
        Runtime='python3.12',
        Role=role_arn,
        Handler='lambda_function.lambda_handler',
        Code={'ZipFile': zip_buffer.read()},
        Description='Lambda function that performs mathematical operations'
    )
    print(f"✅ Lambda function '{function_name}' created successfully!")
    print(f"Function ARN: {response['FunctionArn']}")
except lambda_client.exceptions.ResourceConflictException:
    # Function already exists, update it
    zip_buffer.seek(0)
    response = lambda_client.update_function_code(
        FunctionName=function_name,
        ZipFile=zip_buffer.read()
    )
    print(f"✅ Lambda function '{function_name}' updated successfully!")
    print(f"Function ARN: {response['FunctionArn']}")
except Exception as e:
    print(f"❌ Error deploying Lambda function: {e}")
    print("\nNote: If you see an error about the role not being assumable,")
    print("wait a few seconds and try again - IAM changes can take a moment to propagate.")

# Test the Lambda function
function_name = 'math-function'
lambda_client = boto3.client('lambda')

# Test event
test_event = {
    "operation": "add",
    "num1": 10,
    "num2": 5
}

try:
    # Invoke the Lambda function
    response = lambda_client.invoke(
        FunctionName=function_name,
        InvocationType='RequestResponse',
        Payload=json.dumps(test_event)
    )
    
    # Parse the response
    response_payload = json.loads(response['Payload'].read())
    
    if response['StatusCode'] == 200:
        print("✅ Lambda function test successful!")
        print(f"Test Input: {test_event}")
        print(f"Response: {response_payload}")
        
        # Parse the body if it's a string
        if 'body' in response_payload:
            body = json.loads(response_payload['body'])
            print(f"Result: {body.get('result', 'N/A')}")
    else:
        print(f"❌ Lambda function returned status code: {response['StatusCode']}")
        print(f"Response: {response_payload}")
        
except Exception as e:
    print(f"❌ Error testing Lambda function: {e}")
